In [ ]:
# @title SSHX Launcher for Google Colab { display-mode: "form" }

# ADD THIS AT THE TOP OF YOUR COLAB NOTEBOOK
# Prevents kernel from timing out from inside

import time
import threading
import IPython

def keep_alive_cell():
    """Runs in background thread to prevent kernel idle"""
    while True:
        # Execute tiny operation every 5 minutes
        time.sleep(300)
        print(f"[KeepAlive] Still running... {time.strftime('%H:%M:%S')}")

# Start background thread
thread = threading.Thread(target=keep_alive_cell, daemon=True)
thread.start()
print("[KeepAlive] Background keep-alive thread started!")

mode = "Run SSHX Installer in Background" # @param ["Run SSHX and Display Link/Shell","Run SSHX Installer in Background"]

import subprocess
import os

# ── Step 1: Create 'baynar' user and configure sudo ─────────────────────────
user_setup_script = """
if ! id "baynar" &>/dev/null; then
    useradd -m -s /bin/bash baynar
    echo 'baynar:baynar' | chpasswd
    usermod -aG sudo baynar
    echo "baynar ALL=(ALL) NOPASSWD:ALL" | tee /etc/sudoers.d/baynar
    chmod 0440 /etc/sudoers.d/baynar
    echo "✅ User 'baynar' has been created and added to the sudo group."
else
    echo "⚠️  User 'baynar' already exists."
fi
"""

print("👤 Setting up user 'baynar'...")
user_result = subprocess.run(
    ["bash", "-c", user_setup_script],
    capture_output=True,
    text=True
)

# Print user setup output
if user_result.stdout:
    print(user_result.stdout.strip())
if user_result.stderr:
    print(f"⚠️  stderr: {user_result.stderr.strip()}")

if user_result.returncode != 0:
    print("❌ User setup encountered an error.")
else:
    print("✅ User setup complete.\n")

# ── Step 2: Download the SSHX start script ───────────────────────────────────
print("📥 Downloading start.py...")
download = subprocess.run(
    ["wget", "-q",
     "https://raw.githubusercontent.com/Mnyomi2/COLABY/refs/heads/main/start.py",
     "-O", "/content/start.py"],
    capture_output=True,
    text=True
)

if download.returncode != 0:
    print("❌ Failed to download start.py")
    print(download.stderr)
else:
    print("✅ start.py downloaded successfully\n")

# ── Step 3: Verify the file exists ───────────────────────────────────────────
if not os.path.exists("/content/start.py"):
    raise FileNotFoundError("❌ /content/start.py not found after download.")

# ── Step 4: Launch SSHX as 'baynar' user ─────────────────────────────────────
print(f"🚀 Mode: {mode}\n{'─'*45}")

if mode == "Run SSHX and Display Link/Shell":
    p = subprocess.Popen(
        ["sudo", "-i", "-u", "baynar", "python3", "/content/start.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    [print('🔗 '+l.strip()) for l in p.stdout if ('Link:' in l or 'Shell:' in l)]

elif mode == "Run SSHX Installer in Background":
    # Run with nohup in background, capture output to file
    subprocess.run(
        ["sudo", "-i", "-u", "baynar", "bash", "-c",
         "nohup python3 /content/start.py > /tmp/sshx_output.log 2>&1 &"],
        check=True
    )

    print("⏳ Waiting for SSHX to start...")
    time.sleep(3)

    # Read and display Link/Shell from log file
    if os.path.exists("/tmp/sshx_output.log"):
        with open("/tmp/sshx_output.log", "r") as f:
            for line in f:
                if "Link:" in line or "Shell:" in line:
                    print('🔗 ' + line.strip())

    print("\n✅ SSHX is running in the background.")
    print("   The process will keep running as long as this Colab session is alive.")
    print("   Log file: /tmp/sshx_output.log")